# 01 · Calidad de datos y survey

## Objetivo
Medir, por hoja, el punto de operación y las estaciones numéricas del bloque Measured, y guardar esas tablas sin modificar el Excel.

## Alcance
La estadística del caudal usa un solo número por hoja. La del survey usa las estaciones cuya profundidad medida es numérica. No se imputan celdas y no se estima una serie temporal.


In [1]:
from pathlib import Path
import os, sys
aqui = Path.cwd().resolve()
expediente = next(c for c in [aqui, *aqui.parents] if (c / "dataset.md").is_file() and (c / "entrada").is_dir())
os.chdir(expediente)
if str(expediente / "src") not in sys.path:
    sys.path.insert(0, str(expediente / "src"))
libro = expediente / "entrada" / "BES_jun.xlsx"

from ingesta.leer_bes_jun import leer_libro
from ingesta.survey import leer_survey
from analisis.estadisticas import guardar_tablas, tabla_operacion, tabla_survey
from analisis.graficos import guardar_figuras
hojas = leer_libro(libro)
estaciones = leer_survey(libro)
print("hojas_visibles", sum(h["estado_hoja"] == "visible" for h in hojas), "de", len(hojas))
for h in hojas:
    print(
        f"{h['hoja']} pozo={h['pozo']} bsw={h['bsw_numero']} rotulo_bsw={h['bsw_unidad_rotulo']!r} "
        f"fecha_cache={h['fecha_cache']} today={h['fecha_es_today']}"
    )
guardar_tablas(hojas, estaciones)
guardar_figuras(hojas, estaciones)
for fila in tabla_operacion(hojas):
    if fila["variable"] == "caudal_operacion_bfpd":
        print(
            "caudal", fila["hoja"], "n", fila["n"], "BFPD", fila["media"],
            "desviacion_estandar", fila["desviacion_estandar"],
        )
for fila in tabla_survey(estaciones):
    if fila["variable"] == "md_ft":
        print(
            "survey_md", fila["hoja"], "n", fila["n"],
            "min_ft", fila["minimo"], "max_ft", fila["maximo"],
            "paso_min_ft", fila["paso_md_minimo_ft"],
            "paso_mediana_ft", fila["paso_md_mediana_ft"],
            "paso_max_ft", fila["paso_md_maximo_ft"],
        )


hojas_visibles 5 de 5
QF-554H pozo=QF-084 bsw=0.9755 rotulo_bsw='%' fecha_cache=2026-10-17 00:00:00 today=True
QF-545H pozo=QF-301 bsw=0.9766 rotulo_bsw='%' fecha_cache=2026-10-17 00:00:00 today=True
QF-500H pozo=QF-147 bsw=0.9644 rotulo_bsw='%' fecha_cache=2026-10-17 00:00:00 today=True
QF-378HST pozo=QF-005 bsw=0.9557 rotulo_bsw='%' fecha_cache=2026-10-17 00:00:00 today=True
QF-549HST pozo=QF-306 bsw=0.9557 rotulo_bsw='%' fecha_cache=2017-09-06 00:00:00 today=False


caudal QF-554H n 1 BFPD 10301.0 desviacion_estandar None
caudal QF-545H n 1 BFPD 8767.0 desviacion_estandar None
caudal QF-500H n 1 BFPD 7728.0 desviacion_estandar None
caudal QF-378HST n 1 BFPD 7116.0 desviacion_estandar None
caudal QF-549HST n 1 BFPD 5419.0 desviacion_estandar None
survey_md QF-554H n 56 min_ft 0.0 max_ft 4530.0 paso_min_ft 31.0 paso_mediana_ft 78.0 paso_max_ft 437.0
survey_md QF-545H n 68 min_ft 0.0 max_ft 4576.0 paso_min_ft 60.0 paso_mediana_ft 63.0 paso_max_ft 300.0
survey_md QF-500H n 71 min_ft 0.0 max_ft 4637.0 paso_min_ft 30.0 paso_mediana_ft 63.0 paso_max_ft 267.0
survey_md QF-378HST n 59 min_ft 0.0 max_ft 4598.0 paso_min_ft 38.0 paso_mediana_ft 81.0 paso_max_ft 275.0
survey_md QF-549HST n 67 min_ft 0.0 max_ft 4592.0 paso_min_ft 54.0 paso_mediana_ft 63.0 paso_max_ft 362.0


## Criterio de la desviación
Con un solo caudal por hoja, `n=1`. En ese caso el código deja `desviacion_estandar` en blanco: la desviación muestral no está definida. Mínimo, máximo, media y mediana coinciden con ese único valor. Eso describe el punto escrito; no describe una variación en el tiempo.

## Survey
La fila de unidades que sigue al encabezado `Measured` (`Depth (ft)`) no entra en el conteo. Las estaciones numéricas son 56, 68, 71, 59 y 67, de QF-554H a QF-549HST. La profundidad medida empieza en 0 ft y termina entre 4530 ft y 4637 ft. El paso entre estaciones consecutivas no es constante: en QF-554H el mínimo es 31 ft, la mediana 78 ft y el máximo 437 ft. Las otras hojas también tienen un máximo varias veces mayor que su mediana. Esas cifras están en `data_procesados/estadisticas_survey.csv`.

## Unidades y fechas que quedan abiertas
BS&W se imprime con el número de la celda y el rótulo `%`. La fórmula de densidad usa ese número como fracción. Esta celda no convierte el valor. Cuatro fechas cacheadas son 2026-10-17 y provienen de `=TODAY()`. QF-549HST conserva 2017-09-06.

Los 13 `#DIV/0!` de la columna D de QF-549HST están contados en `data_procesados/calidad.md`. Esta celda no los recalcula. No forman parte del caudal ni del survey que aquí se tabula.

## Figuras que escribe esta celda
`reports/figuras/caudal_limites_por_hoja.png` compara el caudal de operación con el mínimo y el máximo de la misma hoja, en BFPD. `head_puntos_60hz.png` muestra los seis puntos de head escritos a 60 Hz, un panel por referencia de bomba, porque las tres hojas S 8000N traen los mismos puntos y las dos S6000N también. `survey_md_tvd_por_hoja.png` dibuja, por hoja, la profundidad medida en ft contra el valor almacenado en Vertical, en ft. `inclinacion_survey_por_hoja.png` es el diagrama de caja de la inclinación, en grados, de cada survey. Ninguna de esas líneas es una curva de fabricante ajustada.
